In [9]:
import smbus
import time
import atexit
import threading

class Pcf:
    def __init__(self, address=0x27, bus=1):
        self.address = address
        self.bus_number = bus
        self.bus = smbus.SMBus(bus)
        self.output_state = 0xFF
        self._write()
        atexit.register(self._cleanup)
        
    def _write(self):
        try:
            self.bus.write_byte(self.address, self.output_state)
            time.sleep(0.01)
        except Exception:
            pass  # <-- здесь обязательно pass
        
    def set_pin(self, pin, value):
        if value == 0:
            self.output_state &= ~(1 << pin)
        else:
            self.output_state |= (1 << pin)
        self._write()
        
    def get_pin(self, pin):
        return (self.output_state >> pin) & 1
        
    def _cleanup(self):
        self.output_state = 0xFF
        self._write()
        print("PCF очищен")


class Motor:
    def __init__(self, pcf, channel, pwm_freq=50):  # Уменьшил частоту до 50 Гц
        self.pcf = pcf
        self.channel = channel
        self._value = 0
        self.pwm_freq = pwm_freq
        self._pwm_thread = None
        self._pwm_running = False
        self._last_pins = (-1, -1)  # Для отслеживания последнего состояния
        
        if channel == 1:
            self.pin_plus = 0
            self.pin_minus = 1
        else:
            self.pin_plus = 3
            self.pin_minus = 2
    
    def _set_pins_safe(self, plus, minus):
        """Безопасная установка пинов с проверкой изменений"""
        if (plus, minus) != self._last_pins:
            try:
                self.pcf.set_pin(self.pin_plus, plus)
                self.pcf.set_pin(self.pin_minus, minus)
                self._last_pins = (plus, minus)
            except:
                # Игнорируем ошибки I2C - они не критичны
                pass
    
    @property
    def value(self):
        return self._value
    
    @value.setter
    def value(self, val):
        self._value = max(-1.0, min(1.0, float(val)))
        self._apply_pwm()
    
    def _apply_pwm(self):
        # Останавливаем предыдущий ШИМ-поток
        self._pwm_running = False
        if self._pwm_thread and self._pwm_thread.is_alive():
            self._pwm_thread.join(0.2)
        
        # Если скорость 0 или очень маленькая - просто стоп
        if abs(self._value) < 0.01:
            self._set_pins_safe(0, 0)
            return
        
        # Определяем пины для включения
        if self._value > 0:
            self.pin_on = self.pin_plus
            self.pin_off = self.pin_minus
            self.forward = True
        else:
            self.pin_on = self.pin_minus
            self.pin_off = self.pin_plus
            self.forward = False
        
        # Запускаем ШИМ в отдельном потоке с пониженной частотой
        self._pwm_running = True
        self._pwm_thread = threading.Thread(target=self._pwm_loop_simple)
        self._pwm_thread.daemon = True
        self._pwm_thread.start()
    
    def _pwm_loop_simple(self):
        """Упрощённый ШИМ-цикл с защитой от ошибок"""
        duty_cycle = abs(self._value)
        
        # Для высоких скоростей используем постоянное включение
        if duty_cycle > 0.9:
            self._set_pins_safe(1 if self.forward else 0, 
                               0 if self.forward else 1)
            while self._pwm_running and abs(self._value) > 0.01:
                time.sleep(0.1)  # Просто ждём
            self._set_pins_safe(0, 0)
            return
        
        period = 1.0 / self.pwm_freq
        on_time = period * duty_cycle
        off_time = period - on_time
        
        while self._pwm_running and abs(self._value) > 0.01:
            try:
                # Включение
                self._set_pins_safe(1 if self.forward else 0,
                                   0 if self.forward else 1)
                time.sleep(on_time)
                
                # Выключение
                self._set_pins_safe(0, 0)
                time.sleep(off_time)
            except:
                # При ошибке просто продолжаем
                pass
        
        self._set_pins_safe(0, 0)
    
    def hard_stop(self):
        """Принудительная остановка через прямую запись в пины"""
        self._value = 0
        self._pwm_running = False
        self.pcf.set_pin(self.pin_plus, 0)
        self.pcf.set_pin(self.pin_minus, 0)
        print(f"Мотор {self.channel} принудительно остановлен")
    
    def stop(self):
        self.value = 0
        # гарантированный аппаратный сброс
        self.pcf.set_pin(self.pin_plus, 0)
        self.pcf.set_pin(self.pin_minus, 0)
   

In [10]:
# Создаём драйвер
pcf = Pcf(address=0x27, bus=1)
# Создаём моторы (можно указать частоту ШИМ)
left_motor = Motor(pcf, 1, pwm_freq=50)
right_motor = Motor(pcf, 2, pwm_freq=50)

# # Теперь value может быть любым от -1 до 1
# print("Едем медленно (30%)")
# left_motor.value = 0.3
# right_motor.value = 0.3
# time.sleep(2)

# print("Едем быстрее (70%)")
# left_motor.value = 0.7
# right_motor.value = 0.7
# time.sleep(2)

# print("Полный газ (100%)")
# left_motor.value = 1.0
# right_motor.value = 1.0
# time.sleep(2)

# print("Разворот на месте (50%)")
# left_motor.value = -0.5
# right_motor.value = 0.5
# time.sleep(3)

# print("Стоп")
# left_motor.stop()
# right_motor.stop()